In [2]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Layer
from tensorflow.keras.models import Model

class Attention(Layer):
    def __init__(self):
        super().__init__()
        self.score = Dense(1, activation="tanh")

    def call(self, inputs):
        scores = self.score(inputs)
        weights = tf.nn.softmax(scores, axis=1)
        context = tf.reduce_sum(inputs * weights, axis=1)
        return context, weights

inputs = Input(shape=(6,))
x = Embedding(1000, 16)(inputs)
x = LSTM(32, return_sequences=True)(x)
context, weights = Attention()(x)
outputs = Dense(1, activation="sigmoid")(context)

model = Model(inputs, outputs)
attention_model = Model(inputs, weights)

model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])

X = tf.random.uniform((100, 6), minval=1, maxval=1000, dtype=tf.int32)
y = tf.cast(tf.reduce_sum(X, axis=1) % 2, tf.float32)

model.fit(X, y, epochs=3, verbose=1)

sample = tf.constant([[12, 45, 23, 78, 5, 9]])
print("Prediction:", model.predict(sample, verbose=0))
print("Attention weights:",
      attention_model.predict(sample, verbose=0).flatten())

Epoch 1/3
4/4 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.4500 - loss: 0.6934
Epoch 2/3
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.6700 - loss: 0.6923 
Epoch 3/3
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.5500 - loss: 0.6913 
Prediction: [[0.4960457]]
Attention weights: [0.16696833 0.16717148 0.1674316  0.16656716 0.16576605 0.16609542]
